In [1]:
""" Chapter 08: Dimensionality Reduction """

# Not only do all these features make training extreamly slow,
# but they can also make it much harder to find a good solution
# This is called "curse of dimensionality"

# Train your system with the original data before considering using dimensionality reduction.
# Reducing the dimensionality of the training data may filter out some noise and unneccsary details and thus result in higher performance,
# But in general it won't it will just speed up training.

# Reducing dimensionality helps data visualization more useful

' Chapter 08: Dimensionality Reduction '

In [2]:
""" The Curse of Dimensionality """

# We are so used to living in three dimension that our intuition fails us when we try to imagine a high-dimensional space
# It turns out that many things behave very different in high-dimensional space.

# There's just plenty of space in high dimension.
# Most training instances are likely to be far away from each other
# a new instance will likely be far away from any trianing instance, makeing predictions much less reliable than in lower dimensions,
# Since they will be based on much larger extrapolations.

# The more dimensions the training set has -> the greater the risk of overfitting it!
# One solution to the curse of dimensionality could be to increase the size of the training set to reach a sufficient density of training instances.

' The Curse of Dimensionality '

In [3]:
""" Main Approaches for Dimensionality Reduction"""

# There're 2 main approaches to dimensionality:
# 1. Projection
# 2. Manifold learning

' Main Approaches for Dimensionality Reduction'

In [5]:
""" Projection """

# Training instances are NOT spread out uniformly across all dimensions.
# Many features are almost constant, while others are highly correlated

# As a result, all training instances lie within or close to a much lower-dimensional subspace of the high-dimensional space.

# All training instances lie close to a plane: this is a lower-dimensional (2D) subspace of the higher-dimensional (3D) space.
# If we project every training instance perpendicularly onto this subspace -> WE get the new 2D dataset.
# 3D -> 2D

' Projection '

In [6]:
""" Manifold Learning """

# In many cases the subspace may twist and turn.

# The Swiss roll is an example of 2D manifold.
# A 2D manifold is a 2D shape that can be bent and twisted in a higher-dimensional space.
# d-dimensional manifold is a part of an n-dimensional space where d < n that locally resembles a d-dimensional hyperplane.

# Many dimensionality reduction algorithms work by modeling the manifold on which the training instances lie:
# This is called manifold learning. it relies on the manifold assumption, also called the manifold hypothesis, which holds, that most real-world high-dimensional datasets lie close to a much lower-dimensional manifold

# The manifold assumption is often accompanies by another implicit assumptions:
# The task at hand classification and regression will be simpler if expressed in the lower-dimensional space of the manifold.

# Reducing the dimensionality of my training set before training a model will usually speed up training but it may not always lead to a better or simpler solution:
# all depends on the datasets

' Manifold Learning '

In [7]:
""" PCA """

# Principal Component Analysis

# The most popular dimensionality reduction algorithm.
# First it identifies the hyperplane that lies closes to the data, and then it project the data onto it.

' PCA '

In [8]:
""" Preserving the Variance """

# I need to choose the right hyperplane first.
# The projection onto the solid line preserves the maximum variance = Top
# While the projection onto the dotted line preserves very little variance = Bottom
# and the projection onot the dashed line preserves as an intermeditate amount of variance = Middle

# Another way to justify this choice is that it is the axis that minimizes the mean square distance between the original dataset and its projection onto that axis.

' Preserving the Variance '

In [1]:
""" Principal Components """

# If it were a higher-dimensional dataset, PCA would also find a third axis, orthogonal to both previous axes, and a forth and a fifth and so on.
# as many axes as the number of dimensions in the dataset.

# If you perturb the training set slightly and run PCA again, the unit vectors may point in the opposite direction as the original vectors.
# However, they will generally still lie on the same axes.

# There is a standard matrix factorization technique called "Singular Value Decomposition (SVD)"
# that can decompose the training set matrix X into the matrix multiplication of three matrics: U, Sigma, Vector

# Scikit-Learn's PCA classes take care of centering the data for you.
# If you implement PCA yourself as in the preceding example,
# or if you use other libraries, don't forget to center the data first!

import numpy as np

# Create a small 3D dataset
np.random.seed(42)
X = np.random.randn(100, 3)

# Centering data
X_centered = X - X.mean(axis=0)

# Make a SVD
U, S, Vt = np.linalg.svd(X_centered)
c1 = Vt[0]
c2 = Vt[1]

In [2]:
""" Projecting Down to d Dimensions """

# The 3D dataset is projected down to the 2D plane defined by the first 2 principal components, preserving a large part of the dataset's variance.
# As a result, the 2D projection looks very much like the original 3D dataset

# Projecting the training set down to d dimensions
# Xd-proj = XWd

# Projection to 2D
w2 = Vt[:2].T
X2D = X_centered @ w2

In [3]:
""" Using Scikit-Learn """

# Applies PCA to reduce the dimensionality of the dataset down to 2 dimensions
# Note that it automatically takes care of centering the data:

from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X2D = pca.fit_transform(X)

In [4]:
# After fitting the PCA transformer to the dataset, its components_attribute holds the transpose of Wd:
# It contain one row for each of the first d principal components.

In [5]:
""" Explained Variance Ratio """

# The ratio indicates the proportion of the dataset's variance that lies along each princial component.

pca.explained_variance_ratio_

array([0.45375329, 0.3221149 ])

In [6]:
""" Choosing the Right Number of Dimensions """

# It's simpler to choose the number of dimensions that add up to a sufficiently large portion ofthe variance = 95%

# Loads and splits the MNIST dataset and performs PCA without reducing dimensionality then computes the minimum number of dimensions required
# to preserve 95% of the training set's variance.

from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', as_frame=True)
X_train, y_train = mnist.data[:60_000], mnist.target[:60_000]
X_test, y_test = mnist.data[60_000:], mnist.target[60_000:]

pca = PCA()
pca.fit(X_train)
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1 # d = 154

In [7]:
# You can set n_components to be a float between 0.0 and 1.0, indicating the ratio of variance you wish to preserve:

pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train)

In [8]:
pca.n_components_

np.int64(154)

In [9]:
# You can tune the number of dimensions as you would any other hyperparameter

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline

# 1. creates a two-step pipeline, first reducing dimensionality using PCA,
# then classifying using a random forest
clf = make_pipeline(PCA(random_state=42),
                    RandomForestClassifier(random_state=42))

param_distrib = {
    "pca__n_components": np.arange(10, 80),
    "randomforestclassifier__n_estimators": np.arange(50, 500)
}

# 2. Uses RandomizedSearchCV to find a good combination of hyperparameters for both PCA and the random foret classifier.
# tuning only 2 hyperparameters, training on just 1000 instances, and running for just 10 iterations.
rnd_search = RandomizedSearchCV(clf, param_distrib, n_iter=10, cv=3,
                                random_state=42)
rnd_search.fit(X_train[:1000], y_train[:1000])

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'pca__n_components': array([10, 11... 78, 79]), 'randomforestclassifier__n_estimators': array([ 50, ...97, 498, 499])}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchang

In [10]:
# The best hyperparameter is
print(rnd_search.best_params_)

{'randomforestclassifier__n_estimators': np.int64(475), 'pca__n_components': np.int64(57)}


In [11]:
""" PCA for Compression """

# After dimensionality reduction, the training set takes up much less space!
# It is possible to decompress the reducted dataset back to 784 dimensions by applying the inverse_transform() method to the PCA projection.
# It won't give the original back but it will likely to be close to the original data (the projection lost 5% variance)

# compressed and then decompressed is called "reconstructed error"

# Try decompress the reduced MNIST dataset back to 784 dimension:
X_recovered = pca.inverse_transform(X_reduced)

In [12]:
""" Randomized PCA """

# Scikit-Learn uses a stochastic algorithm called "randomized PCA" that quickly finds an approximation of the first d principal components
# O(m X d^2) + O(d^3) much faster for the full SVD approach!

rnd_pca = PCA(n_components=154, svd_solver="randomized", random_state=42)
X_reduced = rnd_pca.fit_transform(X_train)

In [13]:
""" Incremental PCA """

# IPCA = incremental PCA algorithms have been developed that allow me to split the training set into mini-batches and feed these in one mini-batch at the time
# Very useful for large training sets and for applying PCA online

# Let's split the MNIST training set into 100 mini-batches and feeds them to Scikit-Learn's Incremental PCA class to reduce the dimensionality of the MNIST dataset down to 154 dimensions.
# I must called the partial_fit() method with each mini-batch, rathe than the fit() method with the whold training set:

from sklearn.decomposition import IncrementalPCA

n_batches = 100
inc_pca = IncrementalPCA(n_components=154)
for X_batch in np.array_split(X_train, n_batches):
    inc_pca.partial_fit(X_batch)

X_reduced = inc_pca.transform(X_train)

/Users/nattawut/Python-Training/ScikitLearn-Keras-TensorFlow/ML_train/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but IncrementalPCA was fitted without feature names
  warnings.warn(


In [15]:
# I can use memory-mapped or memmap file and copy the MNIST training set to it,
# then call flush() to ensure that any data still in the cache gets saved to disk.
# X_train would typically not fit in the memory so i would load it chunk by chunk and save each chunk to the right part of the memmap array

filename = "my_mnist.mmap"
X_mmap = np.memmap(filename, dtype='float32', mode='write', shape=X_train.shape)
X_mmap[:] = X_train # could be a loop, saving the data chunk by chunk
X_mmap.flush()

In [16]:
# I can load the memmap file and use it like a regular NumPy array.
# Use the incremental PCA class to reduce its dimensionality.
# since this algorithm uses only a small part of the array at any given time, memory usage remains under control.
# I can possibly call the usual fit() method 

X_mmap = np.memmap(filename, dtype='float32', mode='readonly').reshape(-1, 784)
batch_size = X_mmap.shape[0] // n_batches
inc_pca = IncrementalPCA(n_components=154, batch_size=batch_size)
inc_pca.fit(X_mmap)

,"n_components n_components: int, default=NoneNumber of components to keep. If ``n_components`` is ``None``,then ``n_components`` is set to ``min(n_samples, n_features)``.",154
,"whiten whiten: bool, default=FalseWhen True (False by default) the ``components_`` vectors are dividedby ``n_samples`` times ``components_`` to ensure uncorrelated outputswith unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimesimprove the predictive accuracy of the downstream estimators bymaking data respect some hard-wired assumptions.",False
,"copy copy: bool, default=TrueIf False, X will be overwritten. ``copy=False`` can be used tosave memory but is unsafe for general use.",True
,"batch_size batch_size: int, default=NoneThe number of samples to use for each batch. Only used when calling``fit``. If ``batch_size`` is ``None``, then ``batch_size``is inferred from the data and set to ``5 * n_features``, to provide abalance between approximation accuracy and memory consumption.",600


In [17]:
# Only the Raw binary data is saved to the disk. I need to specify the data type and shape of the array when i load it.
# If I omit the shape, np.memmap() returns a 1D array.

# For every high-dimensional datasets, PCA can be too slow.
# The target number of dimensions d must not be too large. If I dealing with a dataset with ten thousands of feature or more such as images,
# then training may become much too slow: in this case, you should consider using random projection instead.

In [2]:
""" Random Projection """

# Projects the data to a lower-dimensional space using a random linear projection.
# Very likely to preserve distances fairly well, as was demonstrated mathematically by William B. Johnson and Joram Lindenstrauss in a famous lemma.

# The more dimensions I drop, the more information is lost, and the more distances get distorted.
# Johnson and Lenderstrauss came up with an equation that determines the minimum number of dimensions to preserve in order to ensure
# with high probability that distances won't change by more than a given tolerance.

# Where each item is sampled randomly from a Gaussian distribution with mean 0 and variance 1 / d and use it to project a dataset from n dimensions down to d
# The only thing the algorithm needs to create the random matrix is the dataset's shape. The data itself is not used at all.

from sklearn.random_projection import johnson_lindenstrauss_min_dim
from sklearn.random_projection import GaussianRandomProjection

# In short, It's usually preferable to use this transformer instead of the first one, especially for large or sparse datasets.
# The ratio r of nonzero items in the sparse random matrix is called its "density", it equal to 1/sqrt(n)

# I can set the density hyperparameter to another value if I prefer.
# If I want to perform the inverse transform, you first need  to compute the pseudo-inverse ofthe components matrix using SciPy's pinv() function.

# Random Projection is a simple,fast, memory-efficient and surprisingly powerful dimensionality reduction algorithm that you should keep in mind,
# Especially, when you deal with high-dimensional datasets.




In [3]:
""" LLE """

# Locally linear embedding (LLE)
# A nonlinear dimensionality reduction (NLDR)

# Its a manifold learning technique that does not rely on projections, unlike PCA and random projection.
# LLE works by first measuring how each training instance linearly relates to its nearest neightbors, and then looking for a low-dimensional representation ofthe training set
# Where these local relationships are best preserved.

# This approach makes it particularly good at unrolling twisted manifolds, especially when there is not too much noise.

from sklearn.datasets import make_swiss_roll
from sklearn.manifold import LocallyLinearEmbedding

X_swiss, t = make_swiss_roll(n_samples=1000, noise=0.2, random_state=42)
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10, random_state=42)
X_unrolled = lle.fit_transform(X_swiss)

In [4]:
# The unrolled swiss roll should be a retangle, not this kind of stretched and twisted band.
# Thus, the first step of LLE is the contrained optimization problem.

# Keeping the weights fixed and finding the optimal position of the instances' images in the low-dimensional space
# Computation complexity: O(m log(m)n log(k)) for finding the k-nearest neightbors, O(mnk^3) for optimization the weights,
# and O(dm^2) for construct the low-dimensional representations.

# LLE is quite complex but it can also construct much better low-dimensional representations, especially if the data is nonlinear.

In [5]:
""" Other Dimensionality Reduction Technique """

# sklearn.manifold.MDS = Multidimensional scalling (MDS) reduces dimensionality while trying to preserve the distances between the instances.
# sklearn.manifold.Isomap = Isomap creates a graph by connecting each instance to its nearest neightbors, then reduces dimensionality while trying to preserve the geodesic distances between the instances.
# sklearn.manifold.TSNE = t-distributed stochastic neightbor embedding (t-SNE) reduces dimensionality while trying to keep similar instances close and dissimilar instances part.
# sklearn.discriminant_analysis.LinearDiscriminantAnalysis = Linear discrimant analysis (LDA) is a linear classification algorithm that, during training learns the most discriminatives axes between the class.

' Other Dimensionality Reduction Technique '